#0. 라이브러리 생성

In [ ]:
!pip install catboost lightgbm xgboost scikit-learn pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, log_loss
import matplotlib.font_manager as fm

from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

SEED     = 42
N_FOLDS  = 5
print('라이브러리 임포트 완료')

# 주요 컬럼 설정
TARGET = "임신 성공 여부"
ID_COL = "ID"
RANDOM_STATE = 42

# 한글 폰트 설정
font_candidates = ["Malgun Gothic", "AppleGothic", "NanumGothic"]
available_fonts = {f.name for f in fm.fontManager.ttflist}

for font in font_candidates:
    if font in available_fonts:
        plt.rcParams["font.family"] = font
        break

plt.rcParams["axes.unicode_minus"] = False

# 모던한 다크 테마
sns.set_theme(style="darkgrid")

plt.rcParams.update({
    "figure.facecolor": "#111111",
    "axes.facecolor": "#111111",
    "axes.edgecolor": "#444444",
    "axes.labelcolor": "#F2F2F2",
    "xtick.color": "#EAEAEA",
    "ytick.color": "#EAEAEA",
    "text.color": "#F2F2F2",
    "axes.titlecolor": "#FFFFFF",
    "grid.color": "#333333",
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 11,
})

# seaborn 테마 적용 후 한글 폰트를 다시 고정
for font in font_candidates:
    if font in available_fonts:
        plt.rcParams["font.family"] = font
        break

main_color = "#6EA8FE"
accent_color = "#FFB86B"
sub_color = "#8BE9A6"


print("font:", plt.rcParams["font.family"])

# 1. 데이터 로딩

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
train= pd.read_csv('/content/drive/MyDrive/ai_health_care_5/7. 프로젝트 2/train.csv')
test=pd.read_csv('/content/drive/MyDrive/ai_health_care_5/7. 프로젝트 2/test.csv')
submission=pd.read_csv('/content/drive/MyDrive/ai_health_care_5/7. 프로젝트 2/sample_submission.csv')
print('데이터 로딩 완료')

TARGET = '임신 성공 여부'
ID_COL = 'ID'

print(f'Train: {train.shape}  Test: {test.shape}')
train.head(3)

#2. 데이터 전처리

In [ ]:
# 1. 학습용 feature와 target 분리 (먼저 실행되어야 함)
train_raw = train.copy()
test_raw = test.copy()

y = train_raw[TARGET].copy()
X = train_raw.drop(columns=[TARGET]).copy()
X_test = test_raw.copy()

train_id = X[ID_COL].copy()
test_id = X_test[ID_COL].copy()


# 2. 비율 및 조합 변수 정제 함수 정의
def add_combo_ratio_features_fixed(df):
    df_feat = df.copy()

    # 조합 변수 생성
    df_feat["treatment_x_specific_proc"] = (
        df_feat["시술 유형"].astype("string").fillna("missing") + "__" +
        df_feat["특정 시술 유형"].astype("string").fillna("missing")
    )
    df_feat["age_x_specific_proc"] = (
        df_feat["시술 당시 나이"].astype("string").fillna("missing") + "__" +
        df_feat["특정 시술 유형"].astype("string").fillna("missing")
    )

    # 안전한 비율 변수 생성 함수
    def safe_ratio_clean(numerator, denominator):
        num = pd.to_numeric(df_feat[numerator], errors="coerce").fillna(0)
        den = pd.to_numeric(df_feat[denominator], errors="coerce").fillna(0)
        ratio = num / den.replace(0, np.nan)
        return ratio.replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 1)

    df_feat["transfer_per_created_embryo"] = safe_ratio_clean("이식된 배아 수", "총 생성 배아 수")
    df_feat["stored_per_created_embryo"] = safe_ratio_clean("저장된 배아 수", "총 생성 배아 수")
    df_feat["mixed_per_fresh_oocyte"] = safe_ratio_clean("혼합된 난자 수", "수집된 신선 난자 수")
    df_feat["icsi_oocyte_rate"] = safe_ratio_clean("미세주입된 난자 수", "혼합된 난자 수")

    return df_feat


# 3. 이제 X가 확실히 존재하므로 에러 없이 적용됩니다!
X = add_combo_ratio_features_fixed(X)
X_test = add_combo_ratio_features_fixed(X_test)

print("데이터 분리 및 파생 변수 적용 완료!")
print("X shape:", X.shape)

In [ ]:
# 결측치 현황 확인

missing = (
    train.isna().sum()
    .to_frame("missing_count")
    .assign(
        missing_rate=lambda x: x["missing_count"] / len(train),
        missing_rate_percent=lambda x: x["missing_rate"] * 100,
    )
    .sort_values("missing_count", ascending=False)
)

display(missing.head(30))

plt.figure(figsize=(9, 8))
plot_df = missing.head(20).sort_values("missing_rate_percent")
ax = sns.barplot(data=plot_df, x="missing_rate_percent", y=plot_df.index, color=main_color)
ax.set_title("결측률 상위 20개 변수") # 전체를 보여주면 너무 많아서 상위 20개만 시각화(EDA)
ax.set_xlabel("결측률(%)")
ax.set_ylabel("")

for i, value in enumerate(plot_df["missing_rate_percent"]):
    ax.text(value + 0.5, i, f"{value:.1f}%", va="center", fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# 학습용 feature와 target 분리
train_raw = train.copy()
test_raw = test.copy()

y = train_raw[TARGET].copy()

X = train_raw.drop(columns=[TARGET]).copy()
X_test = test_raw.copy()

train_id = X[ID_COL].copy()
test_id = X_test[ID_COL].copy()

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

assert TARGET not in X.columns
assert TARGET not in X_test.columns
assert list(X.columns) == list(X_test.columns)

In [ ]:
# 학습용 feature와 target 분리
train_raw = train.copy()
test_raw = test.copy()

y = train_raw[TARGET].copy()

X = train_raw.drop(columns=[TARGET]).copy()
X_test = test_raw.copy()

train_id = X[ID_COL].copy()
test_id = X_test[ID_COL].copy()

print("X:", X.shape)
print("y:", y.shape)
print("X_test:", X_test.shape)

assert TARGET not in X.columns
assert TARGET not in X_test.columns
assert list(X.columns) == list(X_test.columns)

In [ ]:
# 모델 입력 feature 정리
X_model = X.drop(columns=[ID_COL], errors="ignore").copy()
X_test_model = X_test.drop(columns=[ID_COL], errors="ignore").copy()

assert TARGET not in X_model.columns
assert TARGET not in X_test_model.columns
assert list(X_model.columns) == list(X_test_model.columns)

cat_cols = X_model.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = [col for col in X_model.columns if col not in cat_cols]

print("X_model:", X_model.shape)
print("X_test_model:", X_test_model.shape)
print("전체 feature 수:", X_model.shape[1])
print("범주형 변수 수:", len(cat_cols))
print("수치형 변수 수:", len(num_cols))

print("\n범주형 변수 예시:")
print(cat_cols[:20])

print("\n수치형 변수 예시:")
print(num_cols[:20])

In [ ]:
# CatBoost용 데이터: 범주형 변수 문자열 처리
X_cat = X_model.copy()
X_test_cat = X_test_model.copy()

for col in cat_cols:
    X_cat[col] = X_cat[col].astype("string").fillna("missing").astype(str)
    X_test_cat[col] = X_test_cat[col].astype("string").fillna("missing").astype(str)

cat_features = cat_cols.copy()

print("CatBoost용 X_cat:", X_cat.shape)
print("CatBoost용 X_test_cat:", X_test_cat.shape)
print("CatBoost cat_features 수:", len(cat_features))

In [ ]:
# LightGBM/XGBoost 및 트리 모델용 데이터: train 기준 Ordinal Encoding
from sklearn.preprocessing import OrdinalEncoder

X_ordinal = X_model.copy()
X_test_ordinal = X_test_model.copy()

ordinal_encoder = None

if len(cat_cols) > 0:
    ordinal_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    X_cat_values = X_ordinal[cat_cols].astype("string").fillna("missing")
    X_test_cat_values = X_test_ordinal[cat_cols].astype("string").fillna("missing")

    X_ordinal[cat_cols] = ordinal_encoder.fit_transform(X_cat_values)
    X_test_ordinal[cat_cols] = ordinal_encoder.transform(X_test_cat_values)

print("Ordinal Encoding 적용 완료")
print("X_ordinal:", X_ordinal.shape)
print("X_test_ordinal:", X_test_ordinal.shape)

In [ ]:
# 선형/MLP 모델용 결측 대체 및 스케일링
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

X_linear_base = X_ordinal.copy()
X_test_linear_base = X_test_ordinal.copy()

num_like_cols = X_linear_base.columns.tolist()

linear_imputer = SimpleImputer(strategy="median")
linear_scaler = StandardScaler()

X_linear_imputed = pd.DataFrame(
    linear_imputer.fit_transform(X_linear_base),
    columns=num_like_cols,
    index=X_linear_base.index,
)

X_test_linear_imputed = pd.DataFrame(
    linear_imputer.transform(X_test_linear_base),
    columns=num_like_cols,
    index=X_test_linear_base.index,
)

X_linear_scaled = pd.DataFrame(
    linear_scaler.fit_transform(X_linear_imputed),
    columns=num_like_cols,
    index=X_linear_imputed.index,
)

X_test_linear_scaled = pd.DataFrame(
    linear_scaler.transform(X_test_linear_imputed),
    columns=num_like_cols,
    index=X_test_linear_imputed.index,
)

print("X_linear_scaled:", X_linear_scaled.shape)
print("X_test_linear_scaled:", X_test_linear_scaled.shape)

In [ ]:
# 학습/검증 데이터 분리 및 모델별 입력 데이터 준비
from sklearn.model_selection import train_test_split, StratifiedKFold

train_idx, valid_idx = train_test_split(
    np.arange(len(X_model)),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_cb = X_cat.iloc[train_idx].copy()
X_valid_cb = X_cat.iloc[valid_idx].copy()
X_test_cb = X_test_cat.copy()

X_train_gbdt = X_ordinal.iloc[train_idx].copy()
X_valid_gbdt = X_ordinal.iloc[valid_idx].copy()
X_test_gbdt = X_test_ordinal.copy()

X_train_linear = X_linear_scaled.iloc[train_idx].copy()
X_valid_linear = X_linear_scaled.iloc[valid_idx].copy()
X_test_linear = X_test_linear_scaled.copy()

y_train = y.iloc[train_idx].copy()
y_valid = y.iloc[valid_idx].copy()

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("X_train_cb:", X_train_cb.shape)
print("X_valid_cb:", X_valid_cb.shape)
print("X_test_cb:", X_test_cb.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

print("\nTrain target ratio:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nValid target ratio:")
print(y_valid.value_counts(normalize=True).sort_index())

In [ ]:
# 모델 평가 함수 정의
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

def evaluate_binary_model(model_name, y_true, pred_proba, threshold=0.5):
    pred_label = (pred_proba >= threshold).astype(int)

    return {
        "model": model_name,
        "ROC-AUC": roc_auc_score(y_true, pred_proba),
        "LogLoss": log_loss(y_true, pred_proba),
        "Accuracy": accuracy_score(y_true, pred_label),
        "Precision": precision_score(y_true, pred_label, zero_division=0),
        "Recall": recall_score(y_true, pred_label, zero_division=0),
        "F1-score": f1_score(y_true, pred_label, zero_division=0),
    }

In [ ]:
# Baseline 모델 학습 및 평가
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

baseline_results = []

# Logistic Regression
logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

logistic_model.fit(X_train_linear, y_train)
logistic_valid_proba = logistic_model.predict_proba(X_valid_linear)[:, 1]

baseline_results.append(
    evaluate_binary_model(
        "Logistic Regression",
        y_valid,
        logistic_valid_proba
    )
)

# Decision Tree
tree_model = DecisionTreeClassifier(
    max_depth=8,
    min_samples_leaf=50,
    class_weight="balanced",
    random_state=RANDOM_STATE,
)

tree_model.fit(X_train_gbdt, y_train)
tree_valid_proba = tree_model.predict_proba(X_valid_gbdt)[:, 1]

baseline_results.append(
    evaluate_binary_model(
        "Decision Tree",
        y_valid,
        tree_valid_proba
    )
)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=30,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

rf_model.fit(X_train_gbdt, y_train)
rf_valid_proba = rf_model.predict_proba(X_valid_gbdt)[:, 1]

baseline_results.append(
    evaluate_binary_model(
        "Random Forest",
        y_valid,
        rf_valid_proba
    )
)

baseline_results_df = pd.DataFrame(baseline_results).set_index("model")
baseline_results_df

In [ ]:
# CatBoost 모델 학습
from catboost import CatBoostClassifier, Pool

cat_train_pool = Pool(
    X_train_cb,
    y_train,
    cat_features=cat_features,
)

cat_valid_pool = Pool(
    X_valid_cb,
    y_valid,
    cat_features=cat_features,
)

cat_model = CatBoostClassifier(
    iterations=1500,
    learning_rate=0.05,
    depth=6,
    eval_metric="AUC",
    random_seed=RANDOM_STATE,
    l2_leaf_reg=5,
    early_stopping_rounds=150,
    verbose=100
)

cat_model.fit(
    cat_train_pool,
    eval_set=cat_valid_pool,
    use_best_model=True,
)

cat_valid_proba = cat_model.predict_proba(cat_valid_pool)[:, 1]

cat_metrics = evaluate_binary_model(
    "CatBoost",
    y_valid,
    cat_valid_proba
)

cat_metrics

In [ ]:
cat_valid_proba = cat_model.predict_proba(cat_valid_pool)[:, 1]

cat_metrics = evaluate_binary_model(
    "CatBoost",
    y_valid,
    cat_valid_proba
)

model_results_df = pd.concat([
    baseline_results_df,
    pd.DataFrame([cat_metrics]).set_index("model")
])

model_results_df

In [ ]:
# LightGBM 모델 학습
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

lgbm_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

lgbm_model.fit(
    X_train_gbdt,
    y_train,
    eval_set=[(X_valid_gbdt, y_valid)],
    eval_metric="auc",
    callbacks=[
        early_stopping(stopping_rounds=200),
        log_evaluation(period=200),
    ],
)

lgbm_valid_proba = lgbm_model.predict_proba(X_valid_gbdt)[:, 1]

lgbm_metrics = evaluate_binary_model(
    "LightGBM",
    y_valid,
    lgbm_valid_proba
)

lgbm_metrics

In [ ]:
# 기존 결과표에 LightGBM 결과 추가
model_results_df = pd.concat([
    model_results_df,
    pd.DataFrame([lgbm_metrics]).set_index("model")
])

model_results_df

In [ ]:
# XGBoost 모델 학습
from xgboost import XGBClassifier

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=3000,
    learning_rate=0.03,
    max_depth=5,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)

xgb_model.fit(
    X_train_gbdt,
    y_train,
    eval_set=[(X_valid_gbdt, y_valid)],
    verbose=200,
)

xgb_valid_proba = xgb_model.predict_proba(X_valid_gbdt)[:, 1]

xgb_metrics = evaluate_binary_model(
    "XGBoost",
    y_valid,
    xgb_valid_proba
)

xgb_metrics

In [ ]:
model_results_df = pd.concat([
    model_results_df,
    pd.DataFrame([xgb_metrics]).set_index("model")
])

model_results_df

In [ ]:
# 모델 성능 비교 시각화용 데이터 정리
model_plot_df = model_results_df.reset_index().copy()

display(model_plot_df)

In [ ]:
# ROC-AUC 비교 시각화
plot_df = model_plot_df.sort_values("ROC-AUC", ascending=True)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=plot_df,
    x="ROC-AUC",
    y="model",
    color=main_color
)

for i, row in plot_df.iterrows():
    ax.text(
        row["ROC-AUC"] + 0.001,
        list(plot_df.index).index(i),
        f'{row["ROC-AUC"]:.4f}',
        va="center",
        fontsize=10
    )

ax.set_title("모델별 ROC-AUC 비교")
ax.set_xlabel("ROC-AUC")
ax.set_ylabel("")
ax.set_xlim(plot_df["ROC-AUC"].min() - 0.005, plot_df["ROC-AUC"].max() + 0.01)

plt.tight_layout()
plt.show()

In [ ]:
# LogLoss 비교 시각화
plot_df = model_plot_df.sort_values("LogLoss", ascending=False)

plt.figure(figsize=(9, 5))
ax = sns.barplot(
    data=plot_df,
    x="LogLoss",
    y="model",
    color=accent_color
)

for i, row in plot_df.iterrows():
    ax.text(
        row["LogLoss"] + 0.001,
        list(plot_df.index).index(i),
        f'{row["LogLoss"]:.4f}',
        va="center",
        fontsize=10
    )

ax.set_title("모델별 LogLoss 비교")
ax.set_xlabel("LogLoss")
ax.set_ylabel("")
ax.set_xlim(plot_df["LogLoss"].min() - 0.005, plot_df["LogLoss"].max() + 0.01)

plt.tight_layout()
plt.show()

In [ ]:
# CatBoost Feature Importance 확인
cat_feature_importance = pd.DataFrame({
    "feature": X_train_cb.columns,
    "importance": cat_model.get_feature_importance(cat_valid_pool),
})

cat_feature_importance = cat_feature_importance.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

display(cat_feature_importance.head(30))

In [ ]:
# Ensemble 후보 가중치 비교
ensemble_results = []

weight_grid = [
    (0.6, 0.3, 0.1),
    (0.5, 0.4, 0.1),
    (0.5, 0.3, 0.2),
    (0.4, 0.4, 0.2),
    (0.4, 0.3, 0.3),
    (0.7, 0.2, 0.1),
    (0.8, 0.1, 0.1),
]

for cat_w, lgbm_w, xgb_w in weight_grid:
    ensemble_proba = (
        cat_w * cat_valid_proba
        + lgbm_w * lgbm_valid_proba
        + xgb_w * xgb_valid_proba
    )

    metrics = evaluate_binary_model(
        f"Ensemble C{cat_w}_L{lgbm_w}_X{xgb_w}",
        y_valid,
        ensemble_proba
    )

    metrics["cat_weight"] = cat_w
    metrics["lgbm_weight"] = lgbm_w
    metrics["xgb_weight"] = xgb_w

    ensemble_results.append(metrics)

ensemble_results_df = pd.DataFrame(ensemble_results).set_index("model")
ensemble_results_df = ensemble_results_df.sort_values("ROC-AUC", ascending=False)

display(ensemble_results_df)

In [ ]:
# 단일 모델과 Ensemble 결과 비교
best_ensemble = ensemble_results_df.head(1)

model_compare_with_ensemble = pd.concat([
    model_results_df,
    best_ensemble.drop(columns=["cat_weight", "lgbm_weight", "xgb_weight"])
])

display(model_compare_with_ensemble.sort_values("ROC-AUC", ascending=False))

In [ ]:
# CatBoost 5-Fold OOF 검증
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score, log_loss

N_SPLITS = 5

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cat_oof_pred = np.zeros(len(X_cat))
cat_test_pred = np.zeros(len(X_test_cat))
cat_fold_results = []
cat_models = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cat, y), 1):
    print(f"\n========== Fold {fold} ==========")

    X_tr = X_cat.iloc[tr_idx].copy()
    X_val = X_cat.iloc[val_idx].copy()
    y_tr = y.iloc[tr_idx].copy()
    y_val = y.iloc[val_idx].copy()

    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    valid_pool = Pool(X_val, y_val, cat_features=cat_features)
    test_pool = Pool(X_test_cat, cat_features=cat_features)

    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.03,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        random_seed=RANDOM_STATE + fold,
        auto_class_weights="Balanced",
        early_stopping_rounds=200,
        verbose=200,
    )

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    val_pred = model.predict_proba(valid_pool)[:, 1]
    test_pred = model.predict_proba(test_pool)[:, 1]

    cat_oof_pred[val_idx] = val_pred
    cat_test_pred += test_pred / N_SPLITS
    cat_models.append(model)

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_logloss = log_loss(y_val, val_pred)

    cat_fold_results.append({
        "fold": fold,
        "ROC-AUC": fold_auc,
        "LogLoss": fold_logloss,
        "best_iteration": model.get_best_iteration(),
    })

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.6f}")

cat_fold_results_df = pd.DataFrame(cat_fold_results)

cat_oof_metrics = evaluate_binary_model(
    "CatBoost 5-Fold OOF",
    y,
    cat_oof_pred
)

display(cat_fold_results_df)
display(pd.DataFrame([cat_oof_metrics]).set_index("model"))

In [ ]:
# LightGBM 5-Fold OOF 학습
import lightgbm as lgb

lgb_oof_pred = np.zeros(len(X_ordinal))
lgb_test_pred = np.zeros(len(X_test_ordinal))
lgb_fold_results = []
lgb_models = []

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_ordinal, y), start=1):
    print(f"\n========== Fold {fold} ==========")

    X_tr = X_ordinal.iloc[train_idx]
    X_val = X_ordinal.iloc[valid_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[valid_idx]

    lgb_model = lgb.LGBMClassifier(
        n_estimators=5000,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight="balanced",
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        objective="binary"
    )

    lgb_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(stopping_rounds=200),
            lgb.log_evaluation(period=200)
        ]
    )

    val_pred = lgb_model.predict_proba(X_val)[:, 1]
    test_pred = lgb_model.predict_proba(X_test_ordinal)[:, 1]

    lgb_oof_pred[valid_idx] = val_pred
    lgb_test_pred += test_pred / skf.n_splits

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_logloss = log_loss(y_val, val_pred)

    lgb_fold_results.append({
        "fold": fold,
        "ROC-AUC": fold_auc,
        "LogLoss": fold_logloss,
        "best_iteration": lgb_model.best_iteration_
    })

    lgb_models.append(lgb_model)

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.6f}")

lgb_fold_results_df = pd.DataFrame(lgb_fold_results)
display(lgb_fold_results_df)

lgb_oof_metrics = evaluate_binary_model("LightGBM 5-Fold OOF", y, lgb_oof_pred)
display(pd.DataFrame([lgb_oof_metrics]).set_index("model"))

In [ ]:
# CatBoost + LightGBM OOF 앙상블 가중치 탐색
ensemble_results = []

for cat_weight in np.arange(0.0, 1.01, 0.05):
    lgb_weight = 1 - cat_weight

    ensemble_oof_pred = (
        cat_weight * cat_oof_pred +
        lgb_weight * lgb_oof_pred
    )

    metrics = evaluate_binary_model(
        f"Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}",
        y,
        ensemble_oof_pred
    )

    metrics["cat_weight"] = cat_weight
    metrics["lgb_weight"] = lgb_weight

    ensemble_results.append(metrics)

ensemble_results_df = (
    pd.DataFrame(ensemble_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(ensemble_results_df)

In [ ]:
# 03-11. 세밀 가중치 탐색
fine_ensemble_results = []

for cat_weight in np.arange(0.60, 0.801, 0.01):
    lgb_weight = 1 - cat_weight

    ensemble_oof_pred = (
        cat_weight * cat_oof_pred +
        lgb_weight * lgb_oof_pred
    )

    metrics = evaluate_binary_model(
        f"Fine Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}",
        y,
        ensemble_oof_pred
    )

    metrics["cat_weight"] = cat_weight
    metrics["lgb_weight"] = lgb_weight
    fine_ensemble_results.append(metrics)

fine_ensemble_results_df = (
    pd.DataFrame(fine_ensemble_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(fine_ensemble_results_df.head(10))

In [ ]:
# 03-12. Rank Average 앙상블

from scipy.stats import rankdata

cat_oof_rank = rankdata(cat_oof_pred) / len(cat_oof_pred)
lgb_oof_rank = rankdata(lgb_oof_pred) / len(lgb_oof_pred)

rank_ensemble_results = []

for cat_weight in np.arange(0.60, 0.801, 0.01):
    lgb_weight = 1 - cat_weight

    rank_oof_pred = (
        cat_weight * cat_oof_rank +
        lgb_weight * lgb_oof_rank
    )

    metrics = evaluate_binary_model(
        f"Rank Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}",
        y,
        rank_oof_pred
    )

    metrics["cat_weight"] = cat_weight
    metrics["lgb_weight"] = lgb_weight
    rank_ensemble_results.append(metrics)

rank_ensemble_results_df = (
    pd.DataFrame(rank_ensemble_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(rank_ensemble_results_df.head(10))

In [ ]:
# 03-13. 개선안 실행 전 현재 사용 가능한 OOF 예측값 확인

required_oof_objects = {
    "cat_oof_pred": "CatBoost 5-Fold OOF 예측값",
    "cat_test_pred": "CatBoost 5-Fold 제출용 예측값",
    "lgb_oof_pred": "LightGBM 5-Fold OOF 예측값",
    "lgb_test_pred": "LightGBM 5-Fold 제출용 예측값",
}

for obj_name, description in required_oof_objects.items():
    if obj_name not in globals():
        raise NameError(f"{obj_name}이 없습니다. 먼저 해당 모델의 5-Fold 셀을 실행하세요. ({description})")

print("기존 OOF 예측값 확인 완료")
print("cat_oof_pred:", cat_oof_pred.shape)
print("lgb_oof_pred:", lgb_oof_pred.shape)
print("cat_test_pred:", cat_test_pred.shape)
print("lgb_test_pred:", lgb_test_pred.shape)

In [ ]:
# XGBoost 5-Fold OOF 학습
from xgboost import XGBClassifier

xgb_oof_pred = np.zeros(len(X_ordinal))
xgb_test_pred = np.zeros(len(X_test_ordinal))
xgb_fold_results = []
xgb_models = []

neg_count = (y == 0).sum()
pos_count = (y == 1).sum()
scale_pos_weight = neg_count / pos_count

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_ordinal, y), start=1):
    print(f"\n========== Fold {fold} ==========")

    X_tr = X_ordinal.iloc[train_idx]
    X_val = X_ordinal.iloc[valid_idx]
    y_tr = y.iloc[train_idx]
    y_val = y.iloc[valid_idx]

    xgb_model = XGBClassifier(
        n_estimators=3000,
        learning_rate=0.025,
        max_depth=4,
        min_child_weight=12,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=8.0,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE + fold,
        n_jobs=-1,
        early_stopping_rounds=200,
    )

    xgb_model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_val, y_val)],
        verbose=200,
    )

    val_pred = xgb_model.predict_proba(X_val)[:, 1]
    test_pred = xgb_model.predict_proba(X_test_ordinal)[:, 1]

    xgb_oof_pred[valid_idx] = val_pred
    xgb_test_pred += test_pred / skf.n_splits

    fold_auc = roc_auc_score(y_val, val_pred)
    fold_logloss = log_loss(y_val, val_pred)

    xgb_fold_results.append({
        "fold": fold,
        "ROC-AUC": fold_auc,
        "LogLoss": fold_logloss,
        "best_iteration": getattr(xgb_model, "best_iteration", None),
    })

    xgb_models.append(xgb_model)

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Fold {fold} LogLoss: {fold_logloss:.6f}")

xgb_fold_results_df = pd.DataFrame(xgb_fold_results)
display(xgb_fold_results_df)

xgb_oof_metrics = evaluate_binary_model("XGBoost 5-Fold OOF", y, xgb_oof_pred)
display(pd.DataFrame([xgb_oof_metrics]).set_index("model"))

In [ ]:
# CatBoost + LightGBM + XGBoost 3-model 가중치 탐색
three_model_results = []

for cat_weight in np.arange(0.40, 0.91, 0.05):
    for lgb_weight in np.arange(0.00, 0.61, 0.05):
        xgb_weight = 1 - cat_weight - lgb_weight

        if xgb_weight < 0:
            continue

        ensemble_oof_pred = (
            cat_weight * cat_oof_pred
            + lgb_weight * lgb_oof_pred
            + xgb_weight * xgb_oof_pred
        )

        metrics = evaluate_binary_model(
            f"Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}_X{xgb_weight:.2f}",
            y,
            ensemble_oof_pred
        )

        metrics["cat_weight"] = cat_weight
        metrics["lgb_weight"] = lgb_weight
        metrics["xgb_weight"] = xgb_weight

        three_model_results.append(metrics)

three_model_results_df = (
    pd.DataFrame(three_model_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(three_model_results_df.head(20))
# 3-model weighted ensemble 제출용 예측값 생성
best_three = three_model_results_df.iloc[0]

best_cat_weight = best_three["cat_weight"]
best_lgb_weight = best_three["lgb_weight"]
best_xgb_weight = best_three["xgb_weight"]

three_model_oof_pred = (
    best_cat_weight * cat_oof_pred
    + best_lgb_weight * lgb_oof_pred
    + best_xgb_weight * xgb_oof_pred
)

three_model_test_pred = (
    best_cat_weight * cat_test_pred
    + best_lgb_weight * lgb_test_pred
    + best_xgb_weight * xgb_test_pred
)

print("Best 3-model weighted ensemble")
print(best_three[["model", "ROC-AUC", "LogLoss", "cat_weight", "lgb_weight", "xgb_weight"]])
print("OOF ROC-AUC:", roc_auc_score(y, three_model_oof_pred))

In [ ]:
# 3-model Rank Average 가중치 탐색
from scipy.stats import rankdata

cat_oof_rank = rankdata(cat_oof_pred) / len(cat_oof_pred)
lgb_oof_rank = rankdata(lgb_oof_pred) / len(lgb_oof_pred)
xgb_oof_rank = rankdata(xgb_oof_pred) / len(xgb_oof_pred)

cat_test_rank = rankdata(cat_test_pred) / len(cat_test_pred)
lgb_test_rank = rankdata(lgb_test_pred) / len(lgb_test_pred)
xgb_test_rank = rankdata(xgb_test_pred) / len(xgb_test_pred)

rank_three_results = []

for cat_weight in np.arange(0.40, 0.91, 0.05):
    for lgb_weight in np.arange(0.00, 0.61, 0.05):
        xgb_weight = 1 - cat_weight - lgb_weight

        if xgb_weight < 0:
            continue

        rank_oof_pred = (
            cat_weight * cat_oof_rank
            + lgb_weight * lgb_oof_rank
            + xgb_weight * xgb_oof_rank
        )

        metrics = evaluate_binary_model(
            f"Rank Ensemble C{cat_weight:.2f}_L{lgb_weight:.2f}_X{xgb_weight:.2f}",
            y,
            rank_oof_pred
        )

        metrics["cat_weight"] = cat_weight
        metrics["lgb_weight"] = lgb_weight
        metrics["xgb_weight"] = xgb_weight

        rank_three_results.append(metrics)

rank_three_results_df = (
    pd.DataFrame(rank_three_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(rank_three_results_df.head(20))
# 3-model Rank Average 제출용 예측값 생성
best_rank_three = rank_three_results_df.iloc[0]

rank_cat_weight = best_rank_three["cat_weight"]
rank_lgb_weight = best_rank_three["lgb_weight"]
rank_xgb_weight = best_rank_three["xgb_weight"]

rank_three_oof_pred = (
    rank_cat_weight * cat_oof_rank
    + rank_lgb_weight * lgb_oof_rank
    + rank_xgb_weight * xgb_oof_rank
)

rank_three_test_pred = (
    rank_cat_weight * cat_test_rank
    + rank_lgb_weight * lgb_test_rank
    + rank_xgb_weight * xgb_test_rank
)

print("Best 3-model rank ensemble")
print(best_rank_three[["model", "ROC-AUC", "LogLoss", "cat_weight", "lgb_weight", "xgb_weight"]])
print("OOF ROC-AUC:", roc_auc_score(y, rank_three_oof_pred))

In [ ]:
# Raw weighted ensemble과 Rank Average ensemble을 다시 조합
hybrid_results = []

for raw_weight in np.arange(0.0, 1.01, 0.05):
    rank_weight = 1 - raw_weight

    hybrid_oof_pred = (
        raw_weight * three_model_oof_pred
        + rank_weight * rank_three_oof_pred
    )

    metrics = evaluate_binary_model(
        f"Hybrid Raw{raw_weight:.2f}_Rank{rank_weight:.2f}",
        y,
        hybrid_oof_pred
    )

    metrics["raw_weight"] = raw_weight
    metrics["rank_weight"] = rank_weight

    hybrid_results.append(metrics)

hybrid_results_df = (
    pd.DataFrame(hybrid_results)
    .sort_values("ROC-AUC", ascending=False)
    .reset_index(drop=True)
)

display(hybrid_results_df)
# Hybrid ensemble 제출용 예측값 생성
best_hybrid = hybrid_results_df.iloc[0]

best_raw_weight = best_hybrid["raw_weight"]
best_rank_weight = best_hybrid["rank_weight"]

hybrid_oof_pred = (
    best_raw_weight * three_model_oof_pred
    + best_rank_weight * rank_three_oof_pred
)

hybrid_test_pred = (
    best_raw_weight * three_model_test_pred
    + best_rank_weight * rank_three_test_pred
)

print("Best hybrid ensemble")
print(best_hybrid[["model", "ROC-AUC", "LogLoss", "raw_weight", "rank_weight"]])
print("OOF ROC-AUC:", roc_auc_score(y, hybrid_oof_pred))

In [ ]:
# 모델별 성능 비교표 정리
final_model_results = model_compare_with_ensemble.copy()
final_model_results = final_model_results.sort_values("ROC-AUC", ascending=False)

display(final_model_results)